# Build a model RDM

A representational dissimilarity matrix (RDM) compares inputs by one layer's activations: one row per input, correlation distance between rows. This uses rsatoolbox when it is installed (`pip install torchlens[neuro]`) and plain torch otherwise.

In [ ]:
import torch
from torch import nn

import torchlens as tl


def tiny_model(seed: int = 0) -> nn.Module:
    """A deterministic three-layer MLP."""

    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 2),
    )


torch.manual_seed(0)
model = tiny_model(seed=7).eval()
x = torch.randn(5, 4)

In [ ]:
log = tl.trace(model, x, save=["relu_1_2"])
try:
    import rsatoolbox

    dataset = tl.bridge.rsatoolbox.dataset(log, "relu_1_2")
    rdm = rsatoolbox.rdm.calc_rdm(dataset, method="correlation").get_matrices()[0]
except ImportError:
    acts = log["relu_1_2"].out.detach().cpu()
    centred = acts - acts.mean(dim=1, keepdim=True)
    normed = centred / centred.norm(dim=1, keepdim=True)
    rdm = (1 - normed @ normed.T).numpy()
rdm.round(3)

Rows and columns are the five inputs; 0 means perfectly correlated, 2 perfectly anti-correlated.